# Assignment #1: The Donut Effect for Philadelphia ZIP Codes

In this assignment, we will practice our `pandas` skills and explore the ["Donut Effect"](https://www.gsb.stanford.edu/faculty-research/publications/donut-effect-how-covid-19-shapes-real-estate) within Philadelphia. The "Donut Effect" describes the following phenomenon: with more flexible working options and pandemic-driven density fears, people left urban dense cores and opted for more space in city suburbs, driving home and rental prices up in the suburbs relative to city centers.

We will be working with [Zillow data](https://www.zillow.com/research/data/) for the Zillow Home Value Index (ZHVI) for Philadelphia ZIP codes. The goal will be to calculate home price appreciation in Philadelphia, comparing those ZIP codes in Center City (the central business district) to those not in Center City.

## 1. Load the data

I've already downloaded the relevant data file and put in the `data/` folder. Let's load it using `pandas`.  

**Note:** Be sure to use a *relative* file path to make it easier to load your data when grading. See [this guide](https://musa-550-fall-2023.github.io/resource/file-paths.html) for more info.

In [ ]:
import pandas as pd

zhvi = pd.read_csv("Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")
zhvi.head()


## 2. Trim the data to just Philadelphia

Select the subset of the dataframe for Philadelphia, PA.

In [ ]:
phila_zhvi = zhvi["City"].isin(["Philadelphia"])
price_phily =  zhvi.loc[phila_zhvi]
price_phily.head()

## 3. Melt the data into tidy format

Let's transform the data from wide to tidy using the `pd.melt()` function. Create a new column in your data called "ZHVI" that holds the ZHVI values.

In [ ]:

def looks_like_a_date(column_name):
    return column_name.startswith("20")

looks_like_a_date("2000-01-31")
list(filter(looks_like_a_date, zhvi.columns))


ZHVI_Philly_tidy = price_phily.melt(
    id_vars=["City", "RegionName"],
    value_vars=list(filter(looks_like_a_date, zhvi.columns)),   #Notice I'm filtering zhvi columns here
    var_name="Date",
    value_name="ZHVI",
)

ZHVI_Philly_tidy.head()

## 4. Split the data for ZIP codes in/outside Center City

To compare home appreciation in Center City vs. outside Center City, we'll need to split the data into two dataframes, one that holds the Center City ZIP codes and one that holds the data for the rest of the ZIP codes in Philadelphia.

To help with this process, I've included a list of ZIP codes that make up the "greater Center City" region of Philadelphia. Use this list to split the melted data into two dataframes.

In [ ]:
greater_center_city_zip_codes = [
    19123,
    19102,
    19103,
    19106,
    19107,
    19109,
    19130,
    19146,
    19147,
]

In [ ]:
in_center_city = ZHVI_Philly_tidy["RegionName"].isin(greater_center_city_zip_codes)

ZHVI_center_city = ZHVI_Philly_tidy.loc[in_center_city]
ZHVI_not_center_city = ZHVI_Philly_tidy.loc[~in_center_city]

print(ZHVI_center_city)


## 5. Compare home value appreciation in Philadelpia

In this step, we'll calculate the average percent increase in ZHVI from March 2020 to March 2022 for ZIP codes in/out of Center City. We'll do this by:

- Writing a function (see the template below) that will calculate the percent increase in ZHVI from March 31, 2020 to March 31, 2022
- Group your data and apply this function to calculate the ZHVI percent change for each ZIP code in Philadelphia. Do this for both of your dataframes from the previous step.
- Calculate the average value across ZIP codes for both sets of ZIP codes and then compare

You should see much larger growth for ZIP codes outside of Center City...the Donut Effect! 

In [ ]:
def calculate_percent_increase(group_df):
    """
    Calculate the percent increase from 2020-03-31 to 2022-03-31.
    
    Note that `group_df` is the DataFrame for each group.
    """
    
    #Create selections for the march 2020 and march 2022 data
    sel_2020 = group_df["Date"] == "2020-03-31"
    sel_2022 = group_df["Date"] == "2022-03-31"

    #Get the data for each month (only 1 row, so squeeze it!)
    Mar2020 = group_df.loc[sel_2020].squeeze()
    Mar2022 = group_df.loc[sel_2022].squeeze()

    # Columns to calculate percent change for
    columns = ["ZHVI"]

     # Return the percent change for both columns
    return 100 * (Mar2022[columns] / Mar2020[columns] - 1)

In [ ]:
Center_city_grouped = ZHVI_center_city.groupby("RegionName")
Not_Center_city_grouped = ZHVI_not_center_city.groupby("RegionName")

Center_city_result = Center_city_grouped.apply(calculate_percent_increase)
Not_Center_city_result = Not_Center_city_grouped.apply(calculate_percent_increase)

Average_Center_city_result = Center_city_result.mean()
Average_Not_Center_city_result = Not_Center_city_result.mean()

print("Average Percent Increase for Center Cities:", Average_Center_city_result)
print("Average Percent Increase for Non-Center Cities:", Average_Not_Center_city_result)
